# ITW Mask Generator — MNIST

Trains an image-conditioned mask generator that maximizes mutual information
via a frozen D3PM prior (`min H(X|Y)` under a sparsity budget).

In [ ]:
from itw import (
    MNISTConfig,
    build_dataloader,
    build_mask_model,
    build_pixel_survival_table,
    evaluate_loader,
    load_d3pm,
    plot_mask_grid,
    save_eval_report,
    train_mask_generator,
)
from itw.masks import apply_masked_observation, gumbel_mask
from itw.train import discretize, forward_mask_logits
import torch

In [ ]:
cfg = MNISTConfig(
    device="cuda",
    mask_arch="spatial",  # image-conditioned; use "mlp" for class-only baseline
    n_epochs=200,
    save_every=10,
)
d3pm = load_d3pm(cfg)
model = build_mask_model(cfg)
dataloader = build_dataloader(cfg)

In [ ]:
# Calibrate sparsity -> timestep from pixel survival under forward diffusion
survival_table = build_pixel_survival_table(d3pm, dataloader, device=cfg.device)

In [ ]:
model, survival_table = train_mask_generator(
    cfg,
    model=model,
    d3pm=d3pm,
    dataloader=dataloader,
    survival_table=survival_table,
)

## Evaluation

In [ ]:
metrics = evaluate_loader(
    d3pm, model, cfg, dataloader, survival_table,
    max_batches=20, fixed_sparsity=0.3,
)
print(metrics)
save_eval_report(metrics, f"{cfg.save_dir}/eval.json")

In [ ]:
x, cond = next(iter(dataloader))
sparsity = torch.full((x.shape[0],), 0.3)
x_disc = discretize(x, cfg.num_classes)
mask_logits = forward_mask_logits(model, cfg, x_disc, cond, sparsity)
mask = gumbel_mask(mask_logits, temperature=0.5, hard=True)
y = apply_masked_observation(x_disc, mask, cfg.num_classes)
plot_mask_grid(x_disc, mask, y, cfg.num_classes, title="MNIST ITW masks")

In [ ]:
# Sparsity sweep: one image per digit, varying sparsity
import matplotlib.pyplot as plt
import numpy as np

model.eval()
sparse_levels = torch.arange(0.1, 1.0, 0.1)
rows = []
for digit in range(10):
    x_d, c_d = next(iter(dataloader))
    x_d = discretize(x_d.to(cfg.device), cfg.num_classes)
    idx = (c_d == digit).nonzero(as_tuple=True)[0]
    if len(idx) == 0:
        continue
    x_one = x_d[idx[0:1]].repeat(len(sparse_levels), 1, 1, 1)
    c_one = c_d[idx[0:1]].repeat(len(sparse_levels)).to(cfg.device)
    mask_logits = forward_mask_logits(model, cfg, x_one, c_one, sparse_levels)
    row_masks = gumbel_mask(mask_logits, temperature=0.5, hard=True).cpu()[:, 0]
    rows.append(torch.hstack(list(row_masks)))

stack = torch.vstack(rows)
plt.figure(figsize=(12, 12))
plt.imshow(stack, cmap="gray")
plt.yticks([16 + i * 32 for i in range(len(rows))], np.arange(len(rows)))
plt.ylabel("digit")
plt.xticks([16 + i * 32 for i in range(len(sparse_levels))], np.round(sparse_levels.numpy(), 1))
plt.xlabel("sparsity")
plt.show()